This notevook demonstrates the workflow for generating BB code DEMs. The python scripts simulate_gross_code_sweep.py simulate_gross_code_sweep_slice_parallelize.py use this workflow to perform a sweep of physical model parameters and generate the associated DEMs.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import stim
import numpy as np
import time
from bb_tools import *

import pygsti
import pygsti.tools.errgenproptools as eprop
from pygsti.errorgenpropagation.errorpropagator import ErrorGeneratorPropagator

import pauli_twirled_tools as pt

In [3]:
#DEM construction tools
from pygsti.extras.dem_construction import pygsti_object_builders as obj
from pygsti.extras.dem_construction import dem_tools as dems

In [11]:
#here, we use the BeamSearch decoder, available at https://github.com/ionq-publications/BeamSearchDecoder#. Another easy-to-use option is BPOSD
import os
import sys

module_path = os.path.abspath('../../../../BeamSearchDecoder')
sys.path.append(module_path)

import beamsearch

In [5]:
def make_example_param_dict(p=1, h_scale=0, h_scale_idle=0, spam_error=0):
    h_error_rates_dict = {}
    h_error_rates_dict['Gi'] = {('S','X'):0.001*p,('S','Y'):0.001*p,('S','Z'):0.001*p, ('H','X'):np.sqrt(0.0003*p)*h_scale_idle}
    h_error_rates_dict['Gcnot'] = {('H', 'XX'):np.sqrt(0.001*p)*h_scale}
    h_error_rates_dict['Gh'] = {('S','Z'):0.001*p,('S','Y'):0.001*p,('S','X'):0.001*p}
    h_error_rates_dict['Gcnot'].update({('S','IX'): 0.001*p,('S', 'IY'):0.001*p,('S', 'IZ'):0.001*p,
                                       ('S','XI'): 0.001*p,('S', 'YI'):0.001*p,('S', 'ZI'):0.001*p})

    h_error_rates_dict['Mdefault'] = {('S', 'X'): spam_error}
    h_error_rates_dict['rho0'] = {('S', 'X'): 0}

    return h_error_rates_dict

In [6]:
circuit = build_gross_bb_repeated(rounds=1) #currently set to 1 for demo purposes
dem = circuit.detector_error_model()
sampler = dem.compile_sampler()

In [7]:
shots = 100000
samples = sampler.sample(shots=shots)
print(samples[0].shape)
print(samples[1].shape)

(100000, 144)
(100000, 12)


In [8]:
c, qubit_mapping, measurements, detectors = obj.stim_to_pygsti_circuit(circuit, range(circuit.num_qubits), qubit_relabelling_dict=None, show_qubit_mappings=False, include_idles=False, include_meas_idles=True, include_observables=True)
mr_pspec = obj.create_processor_spec(c, c.line_labels, gates=['Gcnot','Gh','Gi'])
n_qubits = len(c.line_labels)

In [9]:
serial_c = c.serialize()
stim_c_no_extras = obj.pygsti_c_to_stim(serial_c)
tableau = stim.Tableau.from_circuit(stim_c_no_extras, ignore_measurement=True) 
inverse_tableau = tableau.inverse()
sim = stim.TableauSimulator()
sim.set_inverse_tableau(inverse_tableau)

In [10]:
#build error model
h_error_rates_dict = make_example_param_dict(p=2, h_scale=-0.6, h_scale_idle=0.2, spam_error=0.001)
h_model = obj.build_model(h_error_rates_dict, mr_pspec, oneQ_gate_names=['Gh','Gi'], twoQ_gate_names=['Gcnot'], meas_err=0.001)

In [11]:
dets_as_pauli_strings = [dems.get_detector_as_parity(d, measurements, n_qubits) for d in detectors]

In [12]:
###Errorprop setup
h_egp = ErrorGeneratorPropagator(h_model)
time1 = time.time()
eoc_eeg = h_egp.propagate_errorgens_bch(serial_c, bch_order=1, include_spam=True)
time2 = time.time()
print(f'{time2-time1} to propagate errors')

8.117501974105835 to propagate errors


In [15]:
#Build DEM
total_dem = dems.generate_dem_higher_order(dets_as_pauli_strings, eoc_eeg, sim, zassenhaus_order=1, add_type='add')
dem_str = dems.format_dem_stim(total_dem, n_logical=12)

TypeError: unsupported operand type(s) for ** or pow(): 'stim._stim_polyfill.TableauSimulator' and 'int'

In [ ]:
dem = stim.DetectorErrorModel(dem_str)

In [ ]:
decoder = beamsearch.BeamSearch(dem)

In [ ]:
sampler = dem.compile_sampler()
samples = sampler.sample(shots=shots)

In [ ]:
predicted_observables = decoder.decode_batch(samples[0])
num_mistakes = np.sum(np.any(predicted_observables != samples[1], axis=1))

In [ ]:
print(num_mistakes/shots)

## Generating a DEM for the Pauli twirled model

In [ ]:
error_pm_dict = {'Gcnot': h_model.operation_blks['gates'][pygsti.baseobjs.label.Label('Gcnot',(147,216))].factorops[1].to_dense(),
                'Gh': h_model.operation_blks['gates'][pygsti.baseobjs.label.Label('Gh',(147))].factorops[1].to_dense(),
                'Gi': h_model.operation_blks['gates'][pygsti.baseobjs.label.Label('Gi',(147))].factorops[1].to_dense()}

In [ ]:
pt_dem, stimc_noisy = pt.build_pauli_twirled_model(error_pm_dict, circuit, include_idles=False, include_meas_idles=True, spam_error=0.001, return_c=True)

In [ ]:
pt_decoder = beamsearch.BeamSearch(pt_dem)

In [ ]:
%%time
pt_predicted_observables = pt_decoder.decode_batch(samples[0])
pt_num_mistakes = np.sum(np.any(pt_predicted_observables != samples[1], axis=1))

In [ ]:
print(pt_num_mistakes/shots)

In [ ]:
pt_samples = pt_dem.compile_sampler().sample(shots=10000)

In [ ]:
pt_predicted_observables = pt_decoder.decode_batch(pt_samples[0])
pt_pt_num_mistakes = np.sum(np.any(pt_predicted_observables != pt_samples[1], axis=1))

In [ ]:
print(pt_pt_num_mistakes/shots)